# Integración reproducible - Rol 5, cierre del Día 2

## Objetivo
Conectar el silbido procesado con el cromosoma de ocho genes, sintetizar una flauta y evaluar el audio con las interfaces reales del proyecto. La primera sección verifica las interfaces; la sección final ejecuta una corrida end-to-end pequeña y reproducible del algoritmo genético.

La corrida final guarda un WAV con nombre propio y se niega a sobrescribirlo si ya existe. No ejecuta el factorial ni los experimentos del Día 3.

## Evidencia experimental del Día 3

El factorial de 12 configuraciones y cinco semillas, con baselines pareados, ya se ejecutó y quedó guardado en `resultados/rol5/dia3_factorial/`. Este notebook conserva la prueba de integración end-to-end del Día 2; el análisis de los registros del Día 3, sin volver a evaluar individuos, está en `notebooks/05_rol5_metaevaluacion.ipynb`. STFT y los operadores B usados en el factorial siguen pendientes de ratificación colectiva.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import soundfile as sf

RAIZ = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / "src").is_dir() and (ruta / "datos").is_dir()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio desde el directorio actual.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.configuracion import (
    SEMILLAS_EXPERIMENTALES,
    TAMANOS_POBLACION_EXPERIMENTALES,
    ConfiguracionAG,
    crear_plan_experimental,
)
from src.algoritmo_genetico import (
    calcular_estadisticas,
    ejecutar_ag,
    evaluar_poblacion,
    inicializar_poblacion,
    seleccionar_elites,
)
from src.extraccion import cargar_melodia_csv
from src.fitness import evaluar_audio, evaluar_individuo
from src.representacion import (
    GENES_SCHEMA,
    generar_individuo_aleatorio,
    obtener_nombres_genes,
    validar_individuo,
)
from src.operadores import seleccion_torneo
from src.sintesis import FRECUENCIA_MUESTREO, guardar_wav, sintetizar_instrumento

print(f"Raíz del proyecto: {RAIZ}")
print("Genes:", obtener_nombres_genes())

config_base = ConfiguracionAG()
plan = crear_plan_experimental()
assert len(plan) == 12
campos_acordados = (
    "tam_poblacion",
    "max_generaciones",
    "elitismo",
    "tam_torneo",
    "paciencia_estancamiento",
    "tolerancia_mejora",
)
print("Parámetros base comunicados por Rol 4:", {
    campo: getattr(config_base, campo) for campo in campos_acordados
})
print("Tamaños previstos:", TAMANOS_POBLACION_EXPERIMENTALES)
print("Semillas utilizadas en el factorial guardado:", SEMILLAS_EXPERIMENTALES)
print("Plan factorial mostrado; esta celda no ejecuta corridas:")
print("N, elitismo, probabilidad_mutacion; STFT y chroma se analizan separados")
for indice, configuracion in enumerate(plan, start=1):
    print(
        indice,
        configuracion.tam_poblacion,
        configuracion.elitismo,
        configuracion.probabilidad_mutacion,
    )
import hashlib
import json
import subprocess
import time
from src.evaluacion import construir_registro_corrida


## Datos y conexión con extracción

`cargar_melodia_csv` lee la melodía que Rol 1 ya procesó y guardó en CSV; no vuelve a detectar notas desde el silbido. La referencia se carga como audio mono. El fitness exige audios alineados y con igual número de muestras, por lo que esta comprobación falla explícitamente si las entradas no cumplen ese contrato.

In [2]:
RUTA_MELODIA = RAIZ / "datos" / "procesados" / "melodia_01.csv"
RUTA_REFERENCIA = RAIZ / "datos" / "audio" / "referencias" / "flauta_referencia_01.wav"
notas, duraciones = cargar_melodia_csv(RUTA_MELODIA)
referencia, frecuencia_referencia = sf.read(RUTA_REFERENCIA, dtype="float32")

if referencia.ndim != 1:
    raise ValueError("La referencia debe ser mono para evaluar fitness.")
if frecuencia_referencia != FRECUENCIA_MUESTREO:
    raise ValueError(
        f"Frecuencia inesperada: {frecuencia_referencia} Hz; "
        f"se esperaba {FRECUENCIA_MUESTREO} Hz."
    )
fronteras = np.rint(
    np.concatenate(([0.0], np.cumsum(duraciones))) * frecuencia_referencia
).astype(np.int64)
if referencia.size != int(fronteras[-1]):
    raise ValueError(
        "La melodía y la referencia no tienen igual longitud; "
        "no se recortan ni remuestrean implícitamente."
    )
print(
    f"Eventos: {len(notas)} | muestras: {referencia.size} | "
    f"duración: {referencia.size / frecuencia_referencia:.3f} s"
)

Eventos: 16 | muestras: 59645 | duración: 2.705 s


## Prueba inicial de integración (no experimental)

Se genera un cromosoma válido con una semilla fija, se sintetiza el mismo cromosoma dos veces con semilla fija y se comprueba que el audio sea idéntico. Después se evalúan por separado las métricas reales del módulo de fitness. La celda siguiente comprueba además selección por torneo, inicialización, evaluación de población, elitismo y estadísticas usando el audio de referencia real. Esta celda no ejecuta el ciclo generacional; la corrida completa se realiza en la seccion final. Los valores mostrados aqui son salidas de prueba, no resultados experimentales.

In [3]:
config_integracion = ConfiguracionAG(
    tam_poblacion=3,
    elitismo=2,
    semilla=config_base.semilla,
)
rng_integracion = np.random.default_rng(config_integracion.semilla)
poblacion_integracion = inicializar_poblacion(
    config_integracion,
    rng=rng_integracion,
)
if poblacion_integracion.shape != (config_integracion.tam_poblacion, len(GENES_SCHEMA)):
    raise AssertionError("La inicialización no devolvió la forma de población esperada.")

aptitudes_integracion = evaluar_poblacion(
    poblacion_integracion,
    lambda individuo: evaluar_individuo(
        individuo,
        notas,
        duraciones,
        referencia,
        metrica="stft",
    ),
)
if aptitudes_integracion.shape != (len(poblacion_integracion),):
    raise AssertionError("La evaluación no conservó una aptitud por individuo.")

padre = seleccion_torneo(
    poblacion_integracion,
    aptitudes_integracion,
    k=config_integracion.tam_torneo,
    rng=rng_integracion,
)
elites_integracion = seleccionar_elites(
    poblacion_integracion,
    aptitudes_integracion,
    cantidad=config_integracion.elitismo,
)
estadisticas_integracion = calcular_estadisticas(
    poblacion_integracion,
    aptitudes_integracion,
    generacion=0,
)

assert any(np.array_equal(padre, candidato) for candidato in poblacion_integracion)
assert np.array_equal(elites_integracion[0], poblacion_integracion[np.argmax(aptitudes_integracion)])
assert not np.shares_memory(elites_integracion, poblacion_integracion)
assert np.isclose(estadisticas_integracion.fitness_maximo, np.max(aptitudes_integracion))
assert np.isclose(estadisticas_integracion.fitness_medio, np.mean(aptitudes_integracion))
assert np.isfinite(estadisticas_integracion.diversidad_genetica)

print("Población inicial evaluada con fitness real:", len(poblacion_integracion))
print("Fitness en orden población-individuo:", aptitudes_integracion)
print("Padre seleccionado por torneo k=", config_integracion.tam_torneo)
print("Número de élites independientes:", len(elites_integracion))
print("Estadísticas de la población inicial:", estadisticas_integracion)
print("Pruebas de interfaces disponibles completadas; no se ejecutó ciclo generacional.")

Población inicial evaluada con fitness real: 3
Fitness en orden población-individuo: [0.5150123  0.6453372  0.52458545]
Padre seleccionado por torneo k= 3
Número de élites independientes: 2
Estadísticas de la población inicial: EstadisticasGeneracion(generacion=0, fitness_maximo=0.6453372035351678, fitness_medio=0.5616449842387135, desviacion_fitness=0.05930824529128079, diversidad_genetica=0.2325021127489916)
Pruebas de interfaces disponibles completadas; no se ejecutó ciclo generacional.


In [4]:
cromosoma = generar_individuo_aleatorio(
    rng=np.random.default_rng(config_base.semilla)
)
valido, mensaje = validar_individuo(cromosoma)
if not valido:
    raise ValueError(f"El generador produjo un cromosoma inválido: {mensaje}")

SEMILLA_SINTESIS = 2026
audio_prueba = sintetizar_instrumento(
    notas, duraciones, cromosoma, semilla=SEMILLA_SINTESIS
)
audio_repetido = sintetizar_instrumento(
    notas, duraciones, cromosoma, semilla=SEMILLA_SINTESIS
)
if not np.array_equal(audio_prueba, audio_repetido):
    raise AssertionError("La síntesis no fue reproducible con la misma semilla.")
if audio_prueba.ndim != 1 or audio_prueba.size != referencia.size:
    raise AssertionError("La síntesis no coincide con forma/longitud de la referencia.")

metricas = evaluar_audio(audio_prueba, referencia)
fitness_stft_interfaz = evaluar_individuo(
    cromosoma, notas, duraciones, referencia, metrica="stft"
)
if not np.isclose(fitness_stft_interfaz, metricas["fitness_stft"]):
    raise AssertionError("La interfaz de fitness y la evaluación del audio discrepan.")

print("Cromosoma válido:", valido, "| genes:", len(GENES_SCHEMA))
print("Síntesis reproducible:", True, "| muestras:", audio_prueba.size)
print("Métricas de esta comprobación solamente:", metricas)
print("Fitness STFT por interfaz evaluar_individuo:", fitness_stft_interfaz)


Cromosoma válido: True | genes: 8
Síntesis reproducible: True | muestras: 59645
Métricas de esta comprobación solamente: {'error_stft_db': 9.417011843854251, 'fitness_stft': 0.5150123036653107, 'fitness_croma': 0.906467738862723}
Fitness STFT por interfaz evaluar_individuo: 0.5150123036653107


## Comprobaciones de interfaces previas al ciclo

Las secciones anteriores verifican la carga de la melodía procesada, la referencia real, la representación, la síntesis, el fitness y componentes individuales de la dinámica. La corrida completa y sus comprobaciones reproducibles están en la sección siguiente.

## Corrida end-to-end del Día 2

Se ejecuta el ciclo real con N=10, hasta cinco reemplazos, semilla fija y la función de aptitud STFT del Rol 2. Se repite la misma corrida para comprobar que población final, historial y mejor individuo coincidan exactamente. Se registran las evaluaciones observadas, generaciones, parada, operadores, configuración, tiempo y versiones de datos/código. No se ejecutan configuraciones factoriales.

In [5]:
config_dia2 = ConfiguracionAG(
    tam_poblacion=10,
    max_generaciones=5,
    elitismo=2,
    semilla=2026,
)
operadores_dia2 = {"seleccion": "torneo", "cruce": "blend", "mutacion": "gaussiana"}

inicio_dia2 = time.perf_counter()
resultado_dia2 = ejecutar_ag(
    lambda individuo: evaluar_individuo(
        individuo, notas, duraciones, referencia, metrica="stft"
    ),
    config_dia2,
    **operadores_dia2,
)
tiempo_dia2_s = time.perf_counter() - inicio_dia2

resultado_dia2_repetido = ejecutar_ag(
    lambda individuo: evaluar_individuo(
        individuo, notas, duraciones, referencia, metrica="stft"
    ),
    config_dia2,
    **operadores_dia2,
)

poblacion_final_dia2 = resultado_dia2["poblacion_final"]
fitness_final_dia2 = resultado_dia2["fitness_final"]
historial_mejor_dia2 = np.asarray(resultado_dia2["historial_mejor_fitness"], dtype=float)
assert poblacion_final_dia2.shape == (config_dia2.tam_poblacion, len(GENES_SCHEMA))
assert np.all(np.isfinite(poblacion_final_dia2)) and np.all((poblacion_final_dia2 >= 0.0) & (poblacion_final_dia2 <= 1.0))
assert np.all((resultado_dia2["mejor_individuo"] >= 0.0) & (resultado_dia2["mejor_individuo"] <= 1.0))
assert np.all(np.isfinite(fitness_final_dia2)) and np.all(np.isfinite(historial_mejor_dia2))
assert resultado_dia2["evaluaciones"] == config_dia2.tam_poblacion * (resultado_dia2["generaciones_ejecutadas"] + 1)
assert np.all(np.diff(historial_mejor_dia2) >= 0.0)
np.testing.assert_array_equal(poblacion_final_dia2, resultado_dia2_repetido["poblacion_final"])
np.testing.assert_array_equal(fitness_final_dia2, resultado_dia2_repetido["fitness_final"])
np.testing.assert_array_equal(resultado_dia2["mejor_individuo"], resultado_dia2_repetido["mejor_individuo"])
assert resultado_dia2["mejor_fitness"] == resultado_dia2_repetido["mejor_fitness"]
assert resultado_dia2["historial_mejor_fitness"] == resultado_dia2_repetido["historial_mejor_fitness"]

audio_mejor_dia2 = sintetizar_instrumento(notas, duraciones, resultado_dia2["mejor_individuo"])
metricas_mejor_dia2 = evaluar_audio(audio_mejor_dia2, referencia)
ruta_audio_dia2 = RAIZ / "resultados" / "audios" / "rol5_dia2_mejor.wav"
if ruta_audio_dia2.exists():
    audio_existente_dia2, sr_existente_dia2 = sf.read(ruta_audio_dia2, dtype="float32")
    if sr_existente_dia2 != FRECUENCIA_MUESTREO or audio_existente_dia2.shape != audio_mejor_dia2.shape or not np.allclose(audio_existente_dia2, audio_mejor_dia2, atol=1.0 / 32768.0, rtol=0.0):
        raise FileExistsError(f"Existe un WAV distinto; no se sobrescribe: {ruta_audio_dia2}")
    print("El WAV existente coincide y se mantuvo sin cambios.")
else:
    guardar_wav(ruta_audio_dia2, audio_mejor_dia2)

try:
    commit_dia2 = subprocess.run(["git", "rev-parse", "HEAD"], cwd=RAIZ, check=True, capture_output=True, text=True).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    commit_dia2 = None
hash_datos_dia2 = {
    "melodia_sha256": hashlib.sha256(RUTA_MELODIA.read_bytes()).hexdigest(),
    "referencia_sha256": hashlib.sha256(RUTA_REFERENCIA.read_bytes()).hexdigest(),
}
registro_dia2 = construir_registro_corrida(
    metodo="algoritmo_genetico", semilla=config_dia2.semilla,
    configuracion={**config_dia2.__dict__, "operadores": operadores_dia2},
    resultado=resultado_dia2, tiempo_s=tiempo_dia2_s, metrica="fitness_stft",
)
registro_dia2.update({"commit": commit_dia2, **hash_datos_dia2, "metricas_audio_mejor": metricas_mejor_dia2, "repeticion_evaluaciones": resultado_dia2_repetido["evaluaciones"], "repeticion_mejor_fitness": resultado_dia2_repetido["mejor_fitness"], "audio_sha256": hashlib.sha256(ruta_audio_dia2.read_bytes()).hexdigest()})
print(json.dumps(registro_dia2, indent=2, ensure_ascii=False))
print("Audio WAV creado:", ruta_audio_dia2.name, "| muestras:", audio_mejor_dia2.size)

El WAV existente coincide y se mantuvo sin cambios.
{
  "metodo": "algoritmo_genetico",
  "metrica": "fitness_stft",
  "semilla": 2026,
  "mejor_fitness": 0.7152406169458495,
  "error_stft_db": null,
  "generaciones_ejecutadas": 5,
  "evaluaciones": 60,
  "tiempo_s": 1.0790301999950316,
  "motivo_parada": "max_generaciones",
  "mejor_individuo_json": "[0.7488446904881343, 0.22961800787649678, 0.0, 0.5244600438145768, 0.17171095196985525, 0.5617924734882094, 0.6331098964095849, 0.0]",
  "configuracion_json": "{\"elitismo\": 2, \"max_generaciones\": 5, \"operadores\": {\"cruce\": \"blend\", \"mutacion\": \"gaussiana\", \"seleccion\": \"torneo\"}, \"paciencia_estancamiento\": 10, \"probabilidad_cruce\": 0.9, \"probabilidad_mutacion\": 0.05, \"semilla\": 2026, \"tam_poblacion\": 10, \"tam_torneo\": 3, \"tolerancia_mejora\": 0.0001}",
  "commit": "ae86d4e68faf1cb18660e223af502a8adb7b7fe2",
  "melodia_sha256": "eed2535aaddaa1eb697848f0a30c69cdded32f51840121183b7fb4947d086565",
  "referencia_

## Límites de esta verificación

Esta es una corrida de demostración y una repetición idéntica para validar reproducibilidad; no estima rendimiento estadístico del AG. El factorial y el baseline pareado del Día 3 están guardados en `resultados/rol5/dia3_factorial/` y se analizan en `notebooks/05_rol5_metaevaluacion.ipynb`. La sensibilidad de esta última se refiere a hiperparámetros del AG; no debe confundirse con la sensibilidad de genes del Rol 1. Las decisiones de métrica primaria y combinación de operadores continúan pendientes de ratificación colectiva.